In [5]:
import getpass
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings



# Prompt for OpenAI API key with password masking
print("Please enter your OpenAI API key:")
openai_api_key = getpass.getpass("API Key: ")

# Set as environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key

# Verify it was set
if openai_api_key:
    masked_key = f"{openai_api_key[:7]}...{openai_api_key[-4:]}"
    print(f"✓ API key set successfully: {masked_key}\n")
else:
    print("✗ No API key entered")
    exit()




Please enter your OpenAI API key:
✓ API key set successfully: # Load ...gs )



In [6]:

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# CRITICAL: You used DIFFERENT embedding models!
DB_CONFIG = {
    'db_path_english': './chroma-db_tme_english',
    'db_path_italian': './chroma-db_italian',
    'db_path_latin': './chroma-db_latin',
    'collection_name_english': 'tme_english',
    'collection_name_italian': 'tmi_italian',
    'collection_name_latin': 'tml_latin',
    'embedding_model': 'text-embedding-3-small'  # Same for all now!
}



In [7]:
# Initialize embeddings (same for both)
embeddings = OpenAIEmbeddings(model=DB_CONFIG['embedding_model'])

print("=" * 70)
print("LOADING DATABASES")
print("=" * 70)

# Load English database
vector_store_english = Chroma(
    persist_directory=DB_CONFIG['db_path_english'],
    collection_name=DB_CONFIG['collection_name_english'],
    embedding_function=embeddings
)

# Load Italian database
vector_store_italian = Chroma(
    persist_directory=DB_CONFIG['db_path_italian'],
    collection_name=DB_CONFIG['collection_name_italian'],
    embedding_function=embeddings
)

# Load Latin database
vector_store_latin = Chroma(
    persist_directory=DB_CONFIG['db_path_latin'],
    collection_name=DB_CONFIG['collection_name_latin'],
    embedding_function=embeddings
)

print(f"✓ Loaded English collection: '{DB_CONFIG['collection_name_english']}'")
print(f"✓ Loaded Italian collection: '{DB_CONFIG['collection_name_italian']}'")
print(f"✓ Loaded Latin collection: '{DB_CONFIG['collection_name_latin']}'")
print("=" * 70)     
# Get samples WITH embeddings included
sample_english = vector_store_english.get(
    limit=3,
    include=['embeddings', 'documents', 'metadatas']
)

sample_italian = vector_store_italian.get(
    limit=3,
    include=['embeddings', 'documents', 'metadatas']
)

sample_latin = vector_store_latin.get(
    limit=3,
    include=['embeddings', 'documents', 'metadatas']
)

# Display English database info
print("\n" + "=" * 70)
print("ENGLISH DATABASE")
print("=" * 70)
print(f"Documents in sample: {len(sample_english['ids'])}")

if sample_english['metadatas'] and len(sample_english['metadatas']) > 0:
    print("\n📄 Sample Document 1:")
    for key, value in sample_english['metadatas'][0].items():
        print(f"  {key}: {value}")
    
    print(f"\n📋 All metadata fields: {sorted(sample_english['metadatas'][0].keys())}")
    
    if sample_english['documents']:
        print(f"\n📝 Text preview: {sample_english['documents'][0][:150]}...")
    
    if sample_english['embeddings'] is not None and len(sample_english['embeddings']) > 0:
        print(f"\n🔢 Embedding dimension: {len(sample_english['embeddings'][0])}")

# Display Italian database info
print("\n" + "=" * 70)
print("ITALIAN DATABASE")
print("=" * 70)
print(f"Documents in sample: {len(sample_italian['ids'])}")

if sample_italian['metadatas'] and len(sample_italian['metadatas']) > 0:
    print("\n📄 Sample Document 1:")
    for key, value in sample_italian['metadatas'][0].items():
        print(f"  {key}: {value}")
    
    print(f"\n📋 All metadata fields: {sorted(sample_italian['metadatas'][0].keys())}")
    
    if sample_italian['documents']:
        print(f"\n📝 Text preview: {sample_italian['documents'][0][:150]}...")
    
    if sample_italian['embeddings'] is not None and len(sample_italian['embeddings']) > 0:
        print(f"\n🔢 Embedding dimension: {len(sample_italian['embeddings'][0])}")


# Display Latin database info
print("\n" + "=" * 70)
print("LATIN DATABASE")
print("=" * 70)
print(f"Documents in sample: {len(sample_latin['ids'])}")       


if sample_latin['metadatas'] and len(sample_latin['metadatas']) > 0:
    print("\n📄 Sample Document 1:")
    for key, value in sample_latin['metadatas'][0].items():
        print(f"  {key}: {value}")
    
    print(f"\n📋 All metadata fields: {sorted(sample_latin['metadatas'][0].keys())}")
    
    if sample_latin['documents']:
        print(f"\n📝 Text preview: {sample_latin['documents'][0][:150]}...")
    
    if sample_latin['embeddings'] is not None and len(sample_latin['embeddings']) > 0:
        print(f"\n🔢 Embedding dimension: {len(sample_latin['embeddings'][0])}")

# COMPATIBILITY CHECK
print("\n" + "=" * 70)
print("COMPATIBILITY ANALYSIS")
print("=" * 70)

compatible = True
issues = []
warnings = []

# Check if we have data
if len(sample_english['ids']) == 0:
    issues.append("English database has no documents")
    compatible = False

if len(sample_italian['ids']) == 0:
    issues.append("Italian database has no documents")
    compatible = False

# Check embeddings exist and compare
has_eng_embeddings = sample_english['embeddings'] is not None and len(sample_english['embeddings']) > 0
has_ita_embeddings = sample_italian['embeddings'] is not None and len(sample_italian['embeddings']) > 0

if compatible and has_eng_embeddings and has_ita_embeddings:
    # Compare embedding dimensions
    dim_eng = len(sample_english['embeddings'][0])
    dim_ita = len(sample_italian['embeddings'][0])
    
    print(f"\n🔢 Embedding Dimensions:")
    print(f"   English: {dim_eng}")
    print(f"   Italian: {dim_ita}")
    
    if dim_eng == dim_ita:
        print(f"   ✓ Dimensions MATCH")
    else:
        print(f"   ✗ Dimensions DIFFER")
        issues.append(f"Embedding dimensions differ: {dim_eng} vs {dim_ita}")
        compatible = False
    
    # Compare metadata schemas
    eng_keys = set(sample_english['metadatas'][0].keys())
    ita_keys = set(sample_italian['metadatas'][0].keys())
    
    common = eng_keys & ita_keys
    only_eng = eng_keys - ita_keys
    only_ita = ita_keys - eng_keys
    
    print(f"\n📋 Metadata Schema:")
    print(f"   Common fields ({len(common)}): {sorted(common)}")
    
    if only_eng:
        print(f"   Only in English ({len(only_eng)}): {sorted(only_eng)}")
        warnings.append(f"English has {len(only_eng)} unique metadata fields")
    
    if only_ita:
        print(f"   Only in Italian ({len(only_ita)}): {sorted(only_ita)}")
        warnings.append(f"Italian has {len(only_ita)} unique metadata fields")
    
    if len(common) == len(eng_keys) == len(ita_keys):
        print(f"   ✓ Metadata schemas MATCH perfectly")
    elif len(common) > 0:
        print(f"   ⚠ Metadata schemas have differences but can be harmonized")

# Final verdict
print("\n" + "=" * 70)
print("FINAL VERDICT")
print("=" * 70)

if issues:
    print("✗ DATABASES ARE INCOMPATIBLE\n")
    for issue in issues:
        print(f"  ✗ {issue}")
elif warnings:
    print("⚠ DATABASES ARE COMPATIBLE WITH MINOR DIFFERENCES\n")
    for warning in warnings:
        print(f"  ⚠ {warning}")
    print("\n✓ These databases CAN be merged for your Streamlit app")
    print("  Missing metadata fields will be null for documents that don't have them")
else:
    print("✓ DATABASES ARE FULLY COMPATIBLE\n")
    print(f"  ✓ Same embedding model: {DB_CONFIG['embedding_model']}")
    print(f"  ✓ Same embedding dimensions: {dim_eng}")
    print(f"  ✓ Identical metadata schemas")
    print("\n✓ Ready to merge for your Streamlit app!")

print("=" * 70)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


LOADING DATABASES
✓ Loaded English collection: 'tme_english'
✓ Loaded Italian collection: 'tmi_italian'
✓ Loaded Latin collection: 'tml_latin'

ENGLISH DATABASE
Documents in sample: 3

📄 Sample Document 1:
  title: A Briefe Discourse
  citation: Unknown
  page_range: 1-2
  date: 1614
  filename: RAVBD_TEXT.html
  date_start: 1600
  date_end: 1699
  author: Ravenscroft, Thomas

📋 All metadata fields: ['author', 'citation', 'date', 'date_end', 'date_start', 'filename', 'page_range', 'title']

📝 Text preview: The Definitions and Diuisions of Moode Time, and Prolation in Measurable 
Musick.

MEnsurabilis Musice is defined to be a Harmony of diuers sortes of ...

🔢 Embedding dimension: 1536

ITALIAN DATABASE
Documents in sample: 3

📄 Sample Document 1:
  title: Le istitutioni harmoniche
  date: 1558
  filename: zarins58.html
  citation: Gioseffo Zarlino, Le istitutioni harmoniche (1558). TMI Reading Edition, accessed at https://tmiweb.science.uu.nl/text/reading-edition/zarins58.html
  page_